# LAS CMS — RAG System (Legal Case Assistant)

A lawyer asks: *"What happened in similar domestic violence cases in Karachi?"*  
The system finds relevant past cases and generates an answer grounded in real data.

### Tech stack
| Component | Now (demo) | Production (swap 2 lines) |
|---|---|---|
| LLM | Ollama (local, free) | OpenAI GPT-4 / Claude |
| Embeddings | sentence-transformers (local) | OpenAI text-embedding-3-small |
| Vector store | ChromaDB (local file) | ChromaDB Cloud / Pinecone |

---

## 1. Setup

**Before running this notebook:**

1. Install Ollama from [ollama.com](https://ollama.com/download)
2. Open a terminal and run: `ollama pull llama3.1:8b` (or any model you prefer)
3. Ollama runs automatically in the background on `localhost:11434`

Then run the cell below to install Python packages.

In [2]:
# # Run this ONCE to install required packages
# import subprocess, sys

# packages = [
#     "chromadb",              # Vector store
#     "sentence-transformers", # Local embeddings
#     "openai",                # Ollama-compatible client (also works with OpenAI)
# ]

# for pkg in packages:
#     print(f"Installing {pkg}...")
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# print("\n✓ All packages installed")

In [1]:
import json
import time
from pathlib import Path
from textwrap import dedent

import chromadb
from sentence_transformers import SentenceTransformer
from openai import OpenAI

print("✓ All imports successful")

✓ All imports successful


## 2. Configuration

**This is the ONLY section you change when going to production.**  
Everything below this cell stays the same.

In [3]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CONFIGURATION — Change these for production             ║
# ╚══════════════════════════════════════════════════════════╝

# ── LLM Settings ──────────────────────────────────────────
# DEMO (Ollama - local, free):
LLM_BASE_URL = "http://localhost:11434/v1"
LLM_API_KEY  = "ollama"                    # Ollama doesn't need a real key
LLM_MODEL    = "llama3.1:8b"              # Change to your downloaded model

# PRODUCTION (uncomment ONE of these):
# --- OpenAI ---
# LLM_BASE_URL = "https://api.openai.com/v1"
# LLM_API_KEY  = "sk-your-openai-key-here"
# LLM_MODEL    = "gpt-4o-mini"
#
# --- Anthropic (via OpenAI-compatible proxy) ---
# LLM_BASE_URL = "https://api.anthropic.com/v1"
# LLM_API_KEY  = "sk-ant-your-key-here"
# LLM_MODEL    = "claude-sonnet-4-20250514"

# ── Embedding Settings ────────────────────────────────────
# DEMO (local, free):
EMBEDDING_MODE = "local"                   # "local" or "openai"
LOCAL_EMBED_MODEL = "all-MiniLM-L6-v2"     # Fast, 384 dims, good quality

# PRODUCTION (uncomment):
# EMBEDDING_MODE = "openai"
# OPENAI_EMBED_KEY = "sk-your-openai-key-here"
# OPENAI_EMBED_MODEL = "text-embedding-3-small"

# ── Paths ─────────────────────────────────────────────────
RAG_DATA_PATH = Path("../outputs/rag_ready/cms_cases_rag.jsonl")
CHROMA_DB_PATH = Path("../outputs/chroma_db")

# ── RAG Settings ──────────────────────────────────────────
TOP_K = 8                # Number of chunks to retrieve per query
CHUNK_SCORE_THRESHOLD = 1.5  # Max distance (lower = more relevant)

print("✓ Configuration loaded")
print(f"  LLM: {LLM_MODEL} @ {LLM_BASE_URL}")
print(f"  Embeddings: {EMBEDDING_MODE} ({LOCAL_EMBED_MODEL})")
print(f"  Data: {RAG_DATA_PATH}")

✓ Configuration loaded
  LLM: llama3.1:8b @ http://localhost:11434/v1
  Embeddings: local (all-MiniLM-L6-v2)
  Data: ..\outputs\rag_ready\cms_cases_rag.jsonl


## 3. Load RAG data

Loading the 16,511 chunks from `cms_cases_rag.jsonl` that we prepared in notebook 04.

In [4]:
# Load all chunks
documents = []
with open(RAG_DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            documents.append(json.loads(line))

print(f"✓ Loaded {len(documents):,} chunks")
print(f"  Sample ID: {documents[0]['id']}")
print(f"  Sample text: {documents[0]['text'][:150]}...")
print(f"  Sample metadata keys: {list(documents[0]['metadata'].keys())}")

✓ Loaded 16,511 chunks
  Sample ID: 1_0
  Sample text: Case Type: Civil. District: Karachi Central. Client: Female, age 38. Filed: 2021-07-05. Program: IG. Hearings (6): Hearing 1 (2024-01-22): [NAME_REDAC...
  Sample metadata keys: ['case_id', 'case_type', 'outcome', 'court_level', 'district', 'gender', 'total_hearings', 'length', 'chunk_index', 'source']


In [5]:
# Quick stats
lengths = [len(d['text']) for d in documents]
case_types = {}
outcomes = {}
for d in documents:
    ct = d['metadata'].get('case_type', 'Unknown')
    oc = d['metadata'].get('outcome', 'Unknown')
    case_types[ct] = case_types.get(ct, 0) + 1
    outcomes[oc] = outcomes.get(oc, 0) + 1

print("Chunks by case type:")
for k, v in sorted(case_types.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v:,}")
print(f"\nChunks by outcome:")
for k, v in sorted(outcomes.items(), key=lambda x: -x[1]):
    print(f"  {k}: {v:,}")
print(f"\nAvg chunk length: {sum(lengths)/len(lengths):.0f} chars")

Chunks by case type:
  Family: 6,779
  UNSPECIFIED: 4,571
  Criminal: 3,803
  Civil: 1,293
  Constitutional: 65

Chunks by outcome:
  IN_FAVOUR: 6,189
  PENDING: 5,047
  In Progress: 3,346
  AGAINST: 1,533
  Missing: 270
  TRANSFERRED: 125
  Disposed of: 1

Avg chunk length: 622 chars


## 4. Initialize embedding model

This converts text into mathematical vectors. Similar texts get similar vectors,  
which is how the system finds relevant cases.

In [ ]:
class EmbeddingModel:
    """Unified embedding interface — local or OpenAI, same API."""
    
    def __init__(self, mode="local", local_model=None, openai_key=None, openai_model=None):
        self.mode = mode
        
        if mode == "local":
            print(f"Loading local model: {local_model}...")
            self.model = SentenceTransformer(local_model)
            self.dimension = self.model.get_sentence_embedding_dimension()
            print(f"✓ Local model loaded ({self.dimension} dimensions)")
        
        elif mode == "openai":
            from openai import OpenAI as OAI
            self.client = OAI(api_key=openai_key)
            self.openai_model = openai_model
            self.dimension = 1536  # text-embedding-3-small
            print(f"✓ OpenAI embeddings ready ({self.openai_model})")
    
    def embed(self, texts):
        """Embed a list of texts → list of vectors."""
        if isinstance(texts, str):
            texts = [texts]
        
        if self.mode == "local":
            return self.model.encode(texts, show_progress_bar=len(texts) > 100).tolist()
        
        elif self.mode == "openai":
            response = self.client.embeddings.create(input=texts, model=self.openai_model)
            return [r.embedding for r in response.data]


# Initialize
if EMBEDDING_MODE == "local":
    embedder = EmbeddingModel(mode="local", local_model=LOCAL_EMBED_MODEL)
else:
    embedder = EmbeddingModel(mode="openai", 
                               openai_key=OPENAI_EMBED_KEY, 
                               openai_model=OPENAI_EMBED_MODEL)

# Quick test
test_vec = embedder.embed("test query about domestic violence cases")
print(f"\nTest embedding: {len(test_vec[0])} dimensions ✓")

Loading local model: all-MiniLM-L6-v2...


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

## 5. Build vector store (ChromaDB)

This step embeds all 16,511 chunks and stores them in ChromaDB.  
**First run takes 5-10 minutes** (embedding all chunks). After that, it loads from disk instantly.

In [ ]:
# Initialize ChromaDB (persistent — saves to disk)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))

# Check if collection already exists (skip re-embedding)
existing_collections = [c.name for c in chroma_client.list_collections()]

if "las_cases" in existing_collections:
    collection = chroma_client.get_collection("las_cases")
    print(f"✓ Loaded existing collection: {collection.count():,} chunks")
    print("  (Delete ../outputs/chroma_db/ folder to rebuild)")
else:
    print("Building vector store from scratch...")
    print("This will take 5-10 minutes on CPU. Be patient.\n")
    
    collection = chroma_client.create_collection(
        name="las_cases",
        metadata={"description": "LAS CMS legal cases for RAG"}
    )
    
    # Process in batches (ChromaDB limit + memory management)
    BATCH_SIZE = 200
    total = len(documents)
    
    for i in range(0, total, BATCH_SIZE):
        batch = documents[i:i+BATCH_SIZE]
        
        ids = [d['id'] for d in batch]
        texts = [d['text'] for d in batch]
        metadatas = [d['metadata'] for d in batch]
        
        # Clean metadata — ChromaDB only accepts str, int, float, bool
        clean_metas = []
        for m in metadatas:
            clean = {}
            for k, v in m.items():
                if isinstance(v, (str, int, float, bool)):
                    clean[k] = v
                else:
                    clean[k] = str(v)
            clean_metas.append(clean)
        
        # Embed
        embeddings = embedder.embed(texts)
        
        # Add to collection
        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=clean_metas
        )
        
        done = min(i + BATCH_SIZE, total)
        pct = done / total * 100
        print(f"  [{done:>6,}/{total:,}] {pct:5.1f}%", end="\r")
    
    print(f"\n\n✓ Vector store built: {collection.count():,} chunks")
    print(f"  Saved to: {CHROMA_DB_PATH}")

In [ ]:
# Quick verification — search for something
test_query = "domestic violence case in Karachi"
test_embedding = embedder.embed(test_query)

results = collection.query(
    query_embeddings=test_embedding,
    n_results=3,
)

print(f"Test query: \"{test_query}\"\n")
for i, (doc, meta, dist) in enumerate(zip(
    results['documents'][0], results['metadatas'][0], results['distances'][0]
)):
    print(f"Result {i+1} (distance: {dist:.3f}):")
    print(f"  Case: {meta.get('case_id','?')} | Type: {meta.get('case_type','?')} | Outcome: {meta.get('outcome','?')}")
    print(f"  Text: {doc[:200]}...")
    print()

## 6. Retrieval function

This is the "R" in RAG — given a question, find the most relevant case chunks.

In [ ]:
def retrieve(query, top_k=TOP_K, filters=None):
    """
    Find the most relevant case chunks for a query.
    
    Args:
        query: The lawyer's question
        top_k: Number of chunks to return
        filters: Optional ChromaDB where filter, e.g.
                 {"case_type": "Family"} or
                 {"district": "Karachi Central"}
    
    Returns:
        List of dicts with: text, metadata, distance
    """
    query_embedding = embedder.embed(query)
    
    kwargs = {
        "query_embeddings": query_embedding,
        "n_results": top_k,
    }
    if filters:
        kwargs["where"] = filters
    
    results = collection.query(**kwargs)
    
    chunks = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        chunks.append({
            'text': doc,
            'metadata': meta,
            'distance': dist,
            'case_id': meta.get('case_id', '?'),
            'case_type': meta.get('case_type', '?'),
            'outcome': meta.get('outcome', '?'),
            'district': meta.get('district', '?'),
        })
    
    return chunks


# Test retrieval
results = retrieve("maintenance case where husband refused to pay")
print(f"Found {len(results)} chunks:\n")
for i, r in enumerate(results[:3]):
    print(f"{i+1}. Case #{r['case_id']} | {r['case_type']} | {r['outcome']} | dist: {r['distance']:.3f}")
    print(f"   {r['text'][:150]}...")
    print()

## 7. Connect to LLM (Ollama)

Make sure Ollama is running: open a terminal and type `ollama list` to check.

In [ ]:
# Initialize LLM client (OpenAI-compatible — works with Ollama AND OpenAI)
llm = OpenAI(
    base_url=LLM_BASE_URL,
    api_key=LLM_API_KEY,
)

# Test connection
try:
    response = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": "Say 'Connection successful' in exactly 2 words."}],
        max_tokens=10,
    )
    print(f"✓ LLM connected: {LLM_MODEL}")
    print(f"  Response: {response.choices[0].message.content}")
except Exception as e:
    print(f"✗ LLM connection failed: {e}")
    print(f"\n  Make sure Ollama is running:")
    print(f"    1. Open a terminal")
    print(f"    2. Run: ollama pull {LLM_MODEL}")
    print(f"    3. Ollama starts automatically on localhost:11434")

## 8. The complete RAG pipeline

This combines retrieval + LLM into one function:  
**Question → Find relevant chunks → Build prompt → Generate answer**

In [ ]:
SYSTEM_PROMPT = dedent("""
    You are a legal research assistant for the Legal Aid Society (LAS) of Pakistan.
    
    Your job is to help lawyers by answering questions based on REAL past case data 
    provided to you. You must:
    
    1. ONLY use information from the provided case data — never make up facts
    2. Cite specific case IDs when referencing cases (e.g. "In Case #247...")
    3. Mention outcomes (IN_FAVOUR, AGAINST, PENDING) when relevant
    4. Note patterns across multiple cases when you see them
    5. If the data doesn't contain enough info to answer, say so honestly
    6. Be concise but thorough — lawyers value precision
    
    The case data comes from LAS's Case Management System covering cases from 
    2020-2026 across multiple districts in Pakistan.
""").strip()


def build_context(chunks):
    """Format retrieved chunks into a context string for the LLM."""
    context_parts = []
    seen_cases = set()
    
    for i, chunk in enumerate(chunks):
        case_id = chunk['case_id']
        # Avoid showing duplicate chunks from the same case
        if case_id in seen_cases:
            continue
        seen_cases.add(case_id)
        
        header = f"[Case #{case_id} | {chunk['case_type']} | {chunk['outcome']} | {chunk['district']}]"
        context_parts.append(f"{header}\n{chunk['text']}")
    
    return "\n\n---\n\n".join(context_parts)


def ask(question, top_k=TOP_K, filters=None, show_sources=True):
    """
    The complete RAG pipeline.
    
    Args:
        question: What the lawyer wants to know
        top_k: Number of chunks to retrieve
        filters: Optional metadata filter
        show_sources: Whether to print source cases
    
    Returns:
        The LLM's answer (also printed)
    """
    print(f"Question: {question}")
    if filters:
        print(f"Filters: {filters}")
    print("─" * 60)
    
    # Step 1: Retrieve
    t0 = time.time()
    chunks = retrieve(question, top_k=top_k, filters=filters)
    t_retrieve = time.time() - t0
    
    if not chunks:
        print("No relevant cases found.")
        return None
    
    # Step 2: Build context
    context = build_context(chunks)
    
    # Step 3: Build prompt
    user_message = f"""Based on the following case data from LAS Pakistan, answer this question:

QUESTION: {question}

CASE DATA:
{context}

Provide a clear, evidence-based answer citing specific case numbers."""
    
    # Step 4: Generate answer
    t1 = time.time()
    response = llm.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=0.3,       # Low temp = more factual, less creative
        max_tokens=1000,
    )
    t_generate = time.time() - t1
    
    answer = response.choices[0].message.content
    
    # Print answer
    print(f"\n{answer}")
    
    # Print sources
    if show_sources:
        seen = set()
        print(f"\n{'─' * 60}")
        print(f"Sources ({t_retrieve:.1f}s retrieval + {t_generate:.1f}s generation):")
        for c in chunks:
            if c['case_id'] not in seen:
                seen.add(c['case_id'])
                print(f"  Case #{c['case_id']} | {c['case_type']} | {c['outcome']} | {c['district']}")
    
    return answer


print("✓ RAG pipeline ready")
print("\nUsage:")
print('  ask("What happened in domestic violence cases in Karachi?")')
print('  ask("custody cases", filters={"case_type": "Family"})')

## 9. Try it!

Run any of these example queries, or write your own.

In [ ]:
# Query 1: General question
ask("What are the most common outcomes in domestic violence cases?")

In [ ]:
# Query 2: District-specific
ask("How do family cases in Karachi typically end?",
    filters={"district": "Karachi Central"})

In [ ]:
# Query 3: Outcome patterns
ask("In cases that were decided AGAINST the client, what were the common reasons?",
    filters={"outcome": "AGAINST"})

In [ ]:
# Query 4: Duration / process
ask("How many hearings do typical family court cases have before disposal?")

In [ ]:
# Query 5: Your own question — edit this!
ask("What arguments or evidence helped win maintenance cases?")

## 10. Advanced: filtered search

You can filter by any metadata field before searching.

In [ ]:
# Available filter fields and their values
import pandas as pd

meta_df = pd.read_csv(Path("../outputs/rag_ready/cms_metadata.csv"))
print("Available filters:\n")
for col in ['case_type', 'outcome', 'court_level', 'district', 'gender']:
    if col in meta_df.columns:
        vals = meta_df[col].dropna().value_counts().head(6)
        print(f"  {col}:")
        for v, c in vals.items():
            print(f"    \"{v}\": {c:,} cases")
        print()

In [ ]:
# Example: Only Criminal cases in specific district
ask("What types of criminal cases does LAS handle?",
    filters={"case_type": "Criminal"})

In [ ]:
# Example: Combined filter (ChromaDB $and syntax)
ask("What happened in family cases that were decided against the client?",
    filters={"$and": [{"case_type": "Family"}, {"outcome": "AGAINST"}]})

## 11. Interactive chat mode

Run the cell below for a continuous Q&A session. Type `quit` to exit.

In [ ]:
def chat():
    """Interactive Q&A loop."""
    print("╔══════════════════════════════════════════════════════╗")
    print("║  LAS Legal Case Assistant                            ║")
    print("║  Ask questions about past cases. Type 'quit' to exit ║")
    print("╚══════════════════════════════════════════════════════╝")
    print()
    
    while True:
        question = input("\nYou: ").strip()
        if not question:
            continue
        if question.lower() in ('quit', 'exit', 'q'):
            print("\nGoodbye!")
            break
        
        print()
        try:
            ask(question)
        except Exception as e:
            print(f"Error: {e}")
        print()

# Uncomment to start interactive mode:
# chat()

## 12. Production upgrade guide

When you're ready to upgrade from demo to production, here's exactly what to change.

### Upgrade LLM (Ollama → OpenAI/Claude)

In Section 2, change these 3 lines:
```python
# FROM (Ollama):
LLM_BASE_URL = "http://localhost:11434/v1"
LLM_API_KEY  = "ollama"
LLM_MODEL    = "llama3.1:8b"

# TO (OpenAI):
LLM_BASE_URL = "https://api.openai.com/v1"
LLM_API_KEY  = "sk-your-real-key"
LLM_MODEL    = "gpt-4o-mini"
```

Everything else stays the same — the OpenAI Python client works with both.

### Upgrade embeddings (local → OpenAI)

In Section 2:
```python
# FROM:
EMBEDDING_MODE = "local"

# TO:
EMBEDDING_MODE = "openai"
OPENAI_EMBED_KEY = "sk-your-key"
OPENAI_EMBED_MODEL = "text-embedding-3-small"
```

**Important:** If you change the embedding model, you must **delete the `chroma_db` folder** and rebuild the vector store. Different models produce different vector dimensions.

### Upgrade vector store (local → cloud)

Replace ChromaDB PersistentClient with:
```python
# Pinecone
import pinecone
pc = pinecone.Pinecone(api_key="your-key")
index = pc.Index("las-cases")

# ChromaDB Cloud
chroma_client = chromadb.HttpClient(host="your-server", port=8000)
```

### Add to a web app

Wrap the `ask()` function in a FastAPI endpoint:
```python
from fastapi import FastAPI
app = FastAPI()

@app.post("/ask")
async def query(question: str):
    answer = ask(question, show_sources=False)
    return {"answer": answer}
```